In [ ]:
# ========================================================
# BÀI 2 (TRANG 15-16) - CÂU 7
# Đề bài: Xây dựng cây ID3 dựa trên dữ liệu huấn luyện và sau đó tiến hành
# kiểm thử kết quả của cây bằng ma trận nhầm lẫn. Biểu diễn cây vào trong kết quả thực hiện.
# ========================================================

# 1. BẮT BUỘC: IN HỌ TÊN VÀ MSSV
HO_TEN = "Đào Tiến Đạt"
MSSV = "6451071017"
print(f"HỌ TÊN: {HO_TEN} - MSSV: {MSSV}")
print("=" * 60)

%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import tree, metrics
import os
import pandas as pd
import numpy as np

def find_file(name):
    paths = [
        name,
        os.path.join(r"D:\khai pha du lieu\Lab2\Dataset-20261103T094921Z-1-001\Dataset", name),
        os.path.join("..", "..", "..", "Dataset-20261103T094921Z-1-001", "Dataset", name),
        os.path.join("Dataset", name)
    ]
    for p in paths:
        if os.path.exists(p):
            return p
    return name


# Chuẩn bị dữ liệu
train_data = pd.read_csv(find_file("adult.data.csv"), skipinitialspace=True, na_values='?').dropna()
test_data = pd.read_csv(find_file("adult.test.csv"), skipinitialspace=True, na_values='?').dropna()
num_train = len(train_data)

if 'fnlwgt' in train_data.columns:
    train_data = train_data.drop(columns=['fnlwgt'])
data = pd.concat([train_data, test_data], ignore_index=True)

features = data.drop('income', axis=1)
labels = data['income']
cat_cols = features.select_dtypes(exclude=['int64', 'float64']).columns
features_onehot = pd.get_dummies(features, columns=cat_cols)

X_train = features_onehot[:num_train]
X_test = features_onehot[num_train:]
y_train = labels[:num_train]
y_test = labels[num_train:]

# 2. Xây dựng cây ID3 với criterion='entropy'
clf = tree.DecisionTreeClassifier(criterion='entropy', random_state=42)
clf = clf.fit(X_train, y_train)

# 3. Dự đoán trên tập kiểm thử
tree_pred = clf.predict(X_test)

# 4. Đánh giá độ chính xác và in báo cáo phân lớp
tree_score = metrics.accuracy_score(y_test, tree_pred)
print("Accuracy:", tree_score)
print("Report:\n", metrics.classification_report(y_test, tree_pred))

# 5. Tính toán ma trận nhầm lẫn và biểu diễn lên đồ thị heatmap
tree_cm = metrics.confusion_matrix(y_test, tree_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(tree_cm, annot=True, fmt="d", linewidths=.5, square=True, cmap='Blues_r',
            xticklabels=clf.classes_, yticklabels=clf.classes_)
plt.ylabel('Actual label')
plt.xlabel('Predicted label')
title = 'Decision Tree (ID3) Accuracy Score: 0.0000'.format(tree_score)
plt.title(title, size=15)
plt.show()

# 6. Biểu diễn cây ID3 (hiển thị trực quan tầng đầu và lưu ra file ảnh)
fig, ax = plt.subplots(figsize=(24, 12))
tree.plot_tree(clf, max_depth=3, feature_names=list(X_train.columns), class_names=list(clf.classes_), filled=True, fontsize=10)
plt.title("Biểu diễn cây quyết định ID3 (tối đa độ sâu 3)", fontsize=16)
plt.savefig('decision_tree_ID3.png', dpi=100)
plt.show()
print("Đã lưu hình ảnh cây ID3 vào file decision_tree_ID3.png")
